# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and report the metadata
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.date_published}")
print(f"License: {dataset.metadata.license}")
print(f"Keywords: {', '.join(dataset.metadata.keywords)}\n")

## 2. Data Overview
Review available record sets, their fields, and their `@id`s.

In [ ]:
# List all record sets with their '@id' and names
print("Available record sets and their @id:")
for rset in dataset.metadata.record_sets:
    print(f"  @id: {rset.id} | name: {rset.name}")
    print("    Fields (with @id):")
    for field in rset.fields:
        print(f"      - @id: {field.id} | name: {field.name} | dtype: {field.data_type}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Here we choose the primary survey record set for demonstration. You should use the `@id`s printed above.

In [ ]:
# Gather all record set @ids
record_set_ids = [rset.id for rset in dataset.metadata.record_sets]
# Choose the main record set (the one containing survey responses); update @id if different for your analysis.
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Using record set: {main_record_set_id}")
else:
    raise Exception("No record sets found in metadata.")

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for {record_set_id} with shape {df.shape}")

# Show columns and head for the main record set
print(f"\nColumns in DataFrame for {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
# Select a numeric field for analysis
# You should set the @id of a numeric field from the overview section above.
# Here, we will auto-detect fields with float/int from the metadata and pick one.
numeric_field_id = None
for rset in dataset.metadata.record_sets:
    if rset.id == main_record_set_id:
        for field in rset.fields:
            if field.data_type in ('Integer', 'Float', 'Number'):
                if field.id in dataframes[main_record_set_id].columns:
                    numeric_field_id = field.id
                    print(f"Using numeric field: {field.id} ({field.name}) for EDA")
                    break
        break

if numeric_field_id is None:
    raise Exception("No numeric field found for EDA.")

# Clean and convert numeric column if needed
df = dataframes[main_record_set_id].copy()
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
threshold = df[numeric_field_id].mean()  # or an explicit value, e.g., 10

filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} records\n")
display_cols = [numeric_field_id] + [col for col in df.columns if col != numeric_field_id][:3]
print(filtered_df[display_cols].head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field (e.g., gender, age_group, or education). Find such a field by string type from fields.
group_field_id = None
for rset in dataset.metadata.record_sets:
    if rset.id == main_record_set_id:
        for field in rset.fields:
            if field.data_type == 'Text' and field.id in filtered_df.columns and filtered_df[field.id].nunique() < 10:
                group_field_id = field.id
                print(f"Grouping by field: {field.id} ({field.name})")
                break
        break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the selected numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by a categorical field if detected
if group_field_id:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load a Croissant/AIRR-compliant mental health dataset using `mlcroissant`
- Inspect record sets and their fields via `@id`
- Extract and process tabular data with pandas
- Perform basic EDA, including filtering, normalization, and grouping
- Visualize key data distributions and relationships

This approach enables transparent, reproducible exploration of FAIR datasets. For in-depth analysis, consult the survey codebook and consider ethical and privacy requirements due to potentially sensitive demographic data.